## Step 2: Data Cleaning and Preprocessing

In [ ]:
import matplotlib.pyplot as plt

import pandas as pd
import numpy as np
import requests
import json

In [ ]:
with open("../config/config.json") as config_file: #to load  the config file 
    config = json.load(config_file)

api_token = config["TMDB_API_KEY"] # retrieves the api key from the config file

headers = {
    "Authorization": f"Bearer {api_token}" # sets the authorization header with the api key
}

movie_ids = [0, 299534, 19995, 140607, 299536, 597, 135397,
            420818, 24428, 168259, 99861, 284054, 12445,
            181808, 330457, 351286, 109445, 321612, 260513]

movies_data = []

for movie_id in movie_ids:
    # API endpoint
    url = f"https://api.themoviedb.org/3/movie/{movie_id}"
    
    #sends GET request to the API endpoint with the authorization header for the movie with the given ID 
    response = requests.get(url, headers=headers)
    
    # Check if request was successful
    if response.status_code == 200:
        movies_data.append(response.json())
        print(f"Successfully fetched movie ID: {movie_id}")
    else:
        print(f"Failed to fetch movie ID: {movie_id}, Status code: {response.status_code}")

# Convert the list of movie data to a DataFrame
raw_movies_df = pd.DataFrame(movies_data)

# Save to CSV
raw_movies_df.to_csv("../data/raw/raw_tmdb_movies.csv", index=False)


In [ ]:
#creating dataframe to process movie data
processed_df = raw_movies_df
processed_df

#### Drop irrelevant columns: ['adult', 'imdb_id', 'original_title', 'video', 'homepage'].

In [ ]:
processed_df=processed_df.drop(columns=['adult', 'imdb_id', 'original_title', 'video', 'homepage'])
processed_df.to_csv("../data/processed/processed_df.csv", index=False)
processed_df.columns

#### Evaluate JSON-like columns

In [ ]:
#creating a new dataframe of only json-like columns for evaluation
json_df=processed_df[['belongs_to_collection', 'genres', 'production_countries',
'production_companies', 'spoken_languages']]
#saving only the selected json-like columns as csv
json_df.to_csv("../data/processed/json_df.csv", index=False)

In [ ]:
#confimriming the columns of the new dataframe
json_df.columns

#### Custom Function to Evaluate JSON-like Data, Extract and Clean Key Data Points

In [ ]:
#this only evaluates listed dictionaries and single dictionaries for the sake of the project
# the function takes 3 optional parameters: separator, key, and column_key_map and leverages the explode and normalize f(x)s
# the function returns a DataFrame with JSON-like columns normalized and flattened.
def eval_json(df, column_key_map=None, key=None, separator="|"):
  
    result_df = df.copy()
    column_key_map = column_key_map or {}

    for column in df.columns:
        # Determine the key to use for the current column
        column_key = column_key_map.get(column, key or 'name')  # Use column_key_map, fallback to key, then 'name'

        if df[column].apply(lambda x: isinstance(x, list) and all(isinstance(i, dict) for i in x)).any():
            # Handle list of dictionaries
            exploded = df[column].explode()
            result_df[column] = exploded.map(
                lambda x: x.get(column_key, '') if isinstance(x, dict) else None
            ).groupby(level=0).apply(lambda x: separator.join(x.dropna()))
        elif df[column].apply(lambda x: isinstance(x, dict)).any():
            # Handle single dictionaries
            result_df[column] = df[column].map(
                lambda x: x.get(column_key, '') if isinstance(x, dict) else x
            )
        else:
            # Handle non-dictionary and non-list values (leave as is)
            result_df[column] = df[column]

    return result_df

In [ ]:
#calling the function on json_df
eval_json_df = eval_json(json_df)
eval_json_df


#### 4. Inspect extracted columns using value_counts() to identify anomalies.  

In [ ]:
eval_json_df.value_counts()

In [ ]:
eval_json_df.info()

In [ ]:
processed_df=processed_df.drop(columns=['belongs_to_collection', 'genres', 'production_countries',
'production_companies', 'spoken_languages'])
movie_df=pd.concat([processed_df, eval_json_df], axis=1)
movie_df

In [ ]:
movie_df.info()

## Handling Missing & Incorrect Data 


#### 5. Convert column datatypes:  

In [ ]:

movie_df['budget']=pd.to_numeric(movie_df['budget'], errors='coerce')
movie_df['id']=pd.to_numeric(movie_df['id'], errors='coerce')
movie_df['popularity']=pd.to_numeric(movie_df['popularity'], errors='coerce')
movie_df['vote_average']=pd.to_numeric(movie_df['vote_average'], errors='coerce')
movie_df['release_date']=pd.to_datetime(movie_df['release_date'], errors='coerce')

In [ ]:
movie_df

#### 6. Replace unrealistic values:

#####  Budget/Revenue/Runtime = 0 → Replace with NaN or infer from similar movies.

In [ ]:
movie_df['budget']=movie_df['budget'].replace(0, np.nan)
movie_df['revenue']=movie_df['revenue'].replace(0, np.nan)
movie_df['runtime']=movie_df['runtime'].replace(0, np.nan)

##### Convert 'budget' and 'revenue' to million USD.

In [ ]:
movie_df['revenue'] = movie_df['revenue'] / 1000000
movie_df['budget'] = movie_df['budget'] / 1000000
movie_df=movie_df.rename(columns={'revenue': 'revenue_musd', 'budget': 'budget_musd'})

In [ ]:
movie_df

##### Movies with vote_count = 0 → Analyze their vote_average and adjusting with weighted average of value_count.


In [ ]:
movie_df.loc[movie_df['vote_count'] == 0, 'vote_count'] = movie_df[movie_df['vote_count'] > 0]['vote_count'].mean()
movie_df[['overview', 'tagline']] = movie_df[['overview', 'tagline']].replace(['No Data', 'Null','null','Unknown', 'N/A',-1,''], np.nan)
movie_df

#### 7. Remove duplicates and drop rows with unknown 'id' or 'title'.

In [ ]:
movie_df['origin_country'] = movie_df['origin_country'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else np.nan)
movie_df

In [ ]:
# Remove duplicates from the DataFrame
movie_df = movie_df.drop_duplicates()
movie_df = movie_df.dropna(subset=['id', 'title'])

#### 8. Keep only rows where at least 10 columns have non-NaN values

In [ ]:
movie_df = movie_df.dropna(thresh=10)
movie_df

In [ ]:
movie_df.info()

#### 9. Filter to include only 'Released' movies, then drop 'status'. 

In [ ]:
movie_df=movie_df.loc[movie_df['status'] == 'Released']
movie_df=movie_df.drop(columns=['status'])
movie_df

## Fetching Credits Data to Extract cast_size, director and crew columns

In [ ]:
with open("../config/config.json") as config_file: #to load  the config file 
    config = json.load(config_file)

api_token = config["TMDB_API_KEY"] # retrieves the api key from the config file

headers = {
    "Authorization": f"Bearer {api_token}" # sets the authorization header with the api key
}

movie_ids = [299534, 19995, 140607, 299536, 597, 135397,
            420818, 24428, 168259, 99861, 284054, 12445,
            181808, 330457, 351286, 109445, 321612, 260513]

credits_data = []

for movie_id in movie_ids:
    # API endpoint
    url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits?language=en-US"
    
    #sends GET request to the API endpoint with the authorization header for the movie with the given ID 
    response = requests.get(url, headers=headers)
    
    # Check if request was successful
    if response.status_code == 200:
        credits_data.append(response.json())
        print(f"Successfully fetched credits of movie ID: {movie_id}")
    else:
        print(f"Failed to credits of movie ID: {movie_id}, Status code: {response.status_code}")

# Convert the list of movie data to a DataFrame
credits_df = pd.DataFrame(credits_data)

# Save to CSV
credits_df.to_csv("../data/raw/tmdb_movies_credits.csv", index=False)



In [ ]:
credits_df

#### Evaluating JSON-like columns in the credits DataFrame

In [ ]:
eval_credits_df = eval_json(credits_df, separator="|", key=None, column_key_map=None)
eval_credits_df

#### creating cast_size series

In [ ]:
eval_credits_df['cast_size'] = eval_credits_df.loc[eval_credits_df['cast'].notna(), 'cast'].astype(str).str.split('|').str.len().reindex( eval_credits_df.index, fill_value=0)
eval_credits_df['cast_size'] = eval_credits_df['cast_size'].astype(int)
eval_credits_df['cast_size']

#### creating crew_size series

In [ ]:
eval_credits_df['crew_size'] = eval_credits_df.loc[eval_credits_df['crew'].notna(), 'crew'].astype(str).str.split('|').str.len().reindex( eval_credits_df.index, fill_value=0)
eval_credits_df['crew_size'] = eval_credits_df['crew_size'].astype(int)
eval_credits_df['crew_size']

In [ ]:
eval_credits_df

In [ ]:
eval_credits_df=eval_credits_df.drop(columns=['cast', 'crew','id'])
eval_credits_df

## Reorder & Finalize DataFrame 

#### Merging Movie Data and Credit Data Dataframes

In [ ]:
movie_df=pd.concat([movie_df, eval_credits_df], axis=1)
movie_df

In [ ]:
eval_credits_df = eval_json(credits_df,column_key_map={'crew': 'job'})  #for the crew jobs
eval_credits_df['job']=eval_credits_df['crew']
eval_credits_df=eval_credits_df.drop(columns=['cast','id','crew'])
eval_credits_df

In [ ]:
eval_credits_df1 = eval_json(credits_df)    #for the crew names
eval_credits_df1=eval_credits_df1.drop(columns=['id'])
eval_credits_df1

In [ ]:
eval_jobs_df=pd.concat([eval_credits_df1, eval_credits_df], axis=1)
eval_jobs_df

#### creating director series

In [ ]:
eval_jobs_df['director'] = eval_jobs_df.apply(
    lambda row: row['crew'].split('|')[row['job'].split('|').index('Director')] if 'Director' in row['job'].split('|') else None,
    axis=1
)
eval_jobs_df


In [ ]:
eval_jobs_df=eval_jobs_df.drop(columns=['crew', 'job'])
movie_df=pd.concat([movie_df, eval_jobs_df], axis=1)
movie_df

## Reorder & Finalize DataFrame

#### 10. Reorder columns:

In [ ]:
movie_df=movie_df.reindex(columns=['id', 'title', 'tagline', 'release_date', 'genres', 'belongs_to_collection',
'original_language', 'budget_musd', 'revenue_musd', 'production_companies',
'production_countries', 'vote_count', 'vote_average', 'popularity', 'runtime',
'overview', 'spoken_languages', 'poster_path', 'cast', 'cast_size', 'director', 'crew_size'])
movie_df

In [ ]:
movie_df.to_csv("../data/processed/movie_df.csv", index=False)
movie_df.info()

#### 11. Reset index.

In [ ]:
movie_df=movie_df.reset_index(drop=True)
movie_df

## Step 3: KPI Implementation & Analysis

#### user-defined function to perform operations

In [ ]:
#Ranks a DataFrame based on the specified operation and columns.  
def rank_df(df, operation, columns, ascending=False, filter_col=None, threshold=None):
 
    result = df.copy()
    if filter_col and threshold is not None:
        result = result[result[filter_col] >= threshold]
    
    # Process based on operation type
    if operation == 'sort':
        return result.sort_values(by=columns, ascending=ascending)
        
    elif operation == 'diff':
        col1, col2 = columns
        result['result'] = result[col1] - result[col2]
        return result.sort_values(by='result', ascending=ascending)
        
    elif operation == 'ratio':
        col1, col2 = columns
        result['result'] = result.apply(lambda x: x[col1]/x[col2] if x[col2] > 0 else 0, axis=1)
        return result.sort_values(by='result', ascending=ascending)
    
    else:
        raise ValueError(f"Unknown operation: {operation}")




#### Identify the Best/Worst Performing Movies

In [ ]:
# Highest Revenue
highest_revenue = rank_df(movie_df, 'sort', 'revenue_musd')
# Highest Budget
highest_budget = rank_df(movie_df, 'sort', 'budget_musd')
# Highest Profit (Revenue - Budget)
highest_profit = rank_df(movie_df, 'diff', ('revenue_musd', 'budget_musd'))
# Lowest Profit (Revenue - Budget)
lowest_profit = rank_df(movie_df, 'diff', ('revenue_musd', 'budget_musd'), ascending=True)
# Highest ROI (Revenue / Budget) (only movies with Budget ≥ 10M)
highest_roi = rank_df(movie_df, 'ratio', ('revenue_musd', 'budget_musd'), filter_col='budget_musd', threshold=10)
#  Lowest ROI (only movies with Budget ≥ 10M)
lowest_roi = rank_df(movie_df, 'ratio', ('revenue_musd', 'budget_musd'), ascending=True, filter_col='budget_musd', threshold=10)
#  Most Voted Movies
most_voted = rank_df(movie_df, 'sort', 'vote_count')
# Most Popular Movies
most_popular = rank_df(movie_df, 'sort', 'popularity')

In [ ]:
#dataframe sorted in descending order by revenue
highest_revenue

In [ ]:
#dataframe sorted in descending order by budget
highest_budget

In [ ]:
#dataframe sorted in descending order by profit
highest_profit

In [ ]:
#dataframe sorted in ascending order by profit
lowest_profit 

In [ ]:
# #dataframe sorted in descending by (Revenue / Budget) (only movies with Budget ≥ 10M)
highest_roi 

In [ ]:
# #dataframe sorted in ascending (only movies with Budget ≥ 10M)
lowest_roi

In [ ]:
#Most Voted Movies
most_voted 

## Identify the Best/Worst Performing Movies that require ratings column

#### loading and extracting columns to create accurate rating column

In [ ]:
ratings_df =pd.read_csv('../data/processed/ratings_df.csv' )
ratings_df = ratings_df.drop(columns=['Unnamed: 0'])
ratings_df

####  dividing rating sum by rating count to get average ratings

In [ ]:
ratings_df['rating'] = ratings_df.apply(lambda x: x[1]/x[0] if x[0] > 0 else 0, axis=1)
ratings_df

#### adding ratings column to movie_df

In [ ]:
movie_df = pd.concat([movie_df, ratings_df['rating']], axis=1)
movie_df

In [ ]:
#Highest Rated Movies (only movies with ≥10 votes)
highest_rated = rank_df(movie_df, 'sort', 'rating', filter_col='vote_count', threshold=10)
highest_rated


In [ ]:
lowest_rated = rank_df(movie_df, 'sort', 'rating',  ascending=True, filter_col='vote_count', threshold=10)
lowest_rated

## Advanced Movie Filtering & Search Queries

#### 2. Filter the dataset for specific queries:

#### 1: Find the best-rated Science Fiction Action movies starring Bruce Willis (sorted by Rating - highest to lowest).

In [ ]:

best_rated_movies = highest_rated.query( "'Science Fiction' in genres and 'Action' in genres and 'Bruce Willis' in cast" )
best_rated_movies

#### ranking data by the runtime column

In [ ]:
shortest_runtime = rank_df(movie_df, 'sort', 'runtime', 
                      ascending=True)                    
shortest_runtime

#### Find movies starring Uma Thurman, directed by Quentin Tarantino (sorted by runtime - shortest to longest).

In [ ]:
shortest_runtime_movie=shortest_runtime.query("'Uma Thurman' in cast and 'Quentin Tarantino' in director")
shortest_runtime_movie

## Franchise vs. Standalone Movie Performance

#### 3. Compare movie franchises (belongs_to_collection) vs. standalone movies in terms Mean Revenue, Median ROI, Mean Budget Raised, Mean Popularity, Mean Rating

#### creating  dataframes of franchise and standalone

In [ ]:
franchise_df = movie_df.query("belongs_to_collection.notna()")
standalone_df = movie_df.query("belongs_to_collection.isna()")

In [ ]:
franchise_df

In [ ]:
standalone_df

#### creating  ranked dataframes of franchise and standalone by roi

In [ ]:

franchise_df_roi = rank_df(franchise_df, 'ratio', ('revenue_musd', 'budget_musd'))
standalone_df_roi = rank_df(standalone_df, 'ratio', ('revenue_musd', 'budget_musd'))


In [ ]:
franchise_df_roi

In [ ]:
standalone_df_roi

#### calculating means of franchise and standalone movies data

In [ ]:
standalone_mean = standalone_df_roi['result'].mean().__round__(4)
franchise_mean = franchise_df_roi['result'].mean().__round__(4)

In [ ]:
standalone_mean

In [ ]:
franchise_mean

### custom function to Compares two values and prints which one is larger or if they are equal.
#### label1 and label2 assign respective names to the values being compared for easy identification

In [ ]:

def compare_values(value1, value2, label1="Value 1", label2="Value 2"):
   
    result = (
        f"{label1} ({value1}) is greater than {label2} ({value2})."
        if value1 > value2
        else f"{label2} ({value2}) is greater than {label1} ({value1})."
        if value1 < value2
        else f"{label1} ({value1}) is equal to {label2} ({value2})."
    )
    print(result)


#### Compare movie franchises (belongs_to_collection) vs. standalone movies in terms Mean Revenue

In [ ]:
compare_values(franchise_mean, standalone_mean, label1="Franchises Mean", label2="Standalone Mean")

#### Compare movie franchises (belongs_to_collection) vs. standalone movies in terms Median ROI

#### calculating median roi of franchise and standalone movies data

In [ ]:
standalone_median = standalone_df_roi['result'].median().__round__(4)
franchise_median = franchise_df_roi['result'].median().__round__(4)

In [ ]:
compare_values(franchise_median, standalone_median, label1="Franchises Median", label2="Standalone Median")

#### Compare movie franchises (belongs_to_collection) vs. standalone movies in terms Mean Budget Raised

#### calculating mean budget raised of franchise and standalone movies data

In [ ]:
franchise_budget_mean = franchise_df['budget_musd'].mean().__round__(4)
standalone_budget_mean = standalone_df['budget_musd'].mean().__round__(4)
compare_values(franchise_budget_mean, standalone_budget_mean, label1="Franchises Budget Mean", label2="Standalone Budget Mean")

#### Compare movie franchises (belongs_to_collection) vs. standalone movies in terms Mean Popularity

#### calculating mean popularity of franchise and standalone movies data

In [ ]:
franchise_popularity_mean = franchise_df['popularity'].mean().__round__(4)
standalone_popularity_mean = standalone_df['popularity'].mean().__round__(4)

In [ ]:
compare_values(franchise_popularity_mean, standalone_popularity_mean, label1="Franchises Popularity Mean", label2="Standalone Popularity Mean")

#### Compare movie franchises (belongs_to_collection) vs. standalone movies in terms Mean Rating

#### calculating mean rating of franchise and standalone movies data

In [ ]:
franchise_rating_mean = franchise_df['rating'].mean().__round__(4)
standalone_rating_mean = standalone_df['rating'].mean().__round__(4)

In [ ]:
compare_values(franchise_rating_mean, standalone_rating_mean, label1="Franchises  Mean Rating", label2="Standalone Mean Rating")

## Most Successful Franchises & Directors

#### 4. Find the Most Successful Movie Franchises based on Total number of movies in franchise, Total & Mean Budget, Total & Mean Revenue, Mean Rating

####  based on Total number of movies in franchise

In [ ]:
franchise_analysis = franchise_df.groupby('belongs_to_collection').agg(
    total_movies=('id', 'count')

).sort_values(by='total_movies', ascending=False)

franchise_analysis
franchise_analysis.iloc[0].name

 #### based on Total number of movies in franchise Total number of movies and Mean budget

In [ ]:
franchise_analysis = franchise_df.groupby('belongs_to_collection').agg(

    total_movies=('id', 'count'),
    mean_budget=('budget_musd', 'mean'),
    
).sort_values(by=['total_movies', 'mean_budget'], ascending=False)

franchise_analysis
franchise_analysis.iloc[0].name

 #### based on Total number of movies in franchise Total number of movies and Mean Revenue

In [ ]:
franchise_analysis = franchise_df.groupby('belongs_to_collection').agg(
    total_movies=('id', 'count'),
    mean_revenue=('revenue_musd', 'mean'),
   
).sort_values(by=['total_movies', 'mean_revenue'], ascending=False)

franchise_analysis
franchise_analysis.iloc[0].name

#### based on mean rating

In [ ]:
franchise_analysis = franchise_df.groupby('belongs_to_collection').agg(
    mean_rating=('rating', 'mean')
).sort_values(by='mean_rating', ascending=False)

franchise_analysis
franchise_analysis.iloc[0].name

#### overall based on all conditions: Total number of movies in franchise, Total & Mean Budget, Total & Mean Revenue, Mean Rating

In [ ]:
franchise_analysis = franchise_df.groupby('belongs_to_collection').agg(
    total_movies=('id', 'count'),
    total_budget=('budget_musd', 'sum'),
    mean_budget=('budget_musd', 'mean'),
    total_revenue=('revenue_musd', 'sum'),
    mean_revenue=('revenue_musd', 'mean'),
    mean_rating=('rating', 'mean')
).sort_values(by='total_revenue', ascending=False)

franchise_analysis.iloc[0].name

In [ ]:
franchise_analysis.iloc[0].name

## 5. Find the Most Successful Directors

#### based on Total Number of Movies Directed

In [ ]:
director_analysis = highest_rated.groupby('director').agg(
    total_movies=('id', 'count'),
).sort_values(by='total_revenue', ascending=False)

In [ ]:
director_analysis.iloc[0].name

#### based on Total Revenue

In [ ]:
director_analysis = highest_rated.groupby('director').agg(
    total_revenue=('revenue_musd', 'sum'),
).sort_values(by='total_revenue', ascending=False)

In [ ]:
director_analysis.iloc[0].name

#### based on Mean Rating

In [ ]:
director_analysis = highest_rated.groupby('director').agg(
    mean_rating=('rating', 'mean')
).sort_values(by='total_revenue', ascending=False)

In [ ]:
director_analysis.iloc[0].name

## Step 4: Data Visualization

#### Use Pandas, Matplotlib to visualize

#### Revenue vs. Budget Trends

In [ ]:

plt.figure(figsize=(10, 6))
plt.scatter(highest_revenue['budget_musd'], highest_revenue['revenue_musd'], alpha=0.7, color='blue')
plt.title('Revenue vs Budget (Top Revenue Movies)', fontsize=16)
plt.xlabel('Budget (in million USD)', fontsize=14)
plt.ylabel('Revenue (in million USD)', fontsize=14)
plt.grid(True)
plt.show()

#### ROI Distribution by Genre

In [ ]:
plt.figure(figsize=(12, 8))
highest_roi['genres'] = highest_roi['genres'].fillna('Unknown')  # Handle missing genres
roi_by_genre = highest_roi.explode('genres').groupby('genres')['result'].mean().sort_values(ascending=False)

roi_by_genre.plot(kind='bar', color='green', alpha=0.7)
plt.title('ROI Distribution by Genre', fontsize=16)
plt.xlabel('Genre', fontsize=14)
plt.ylabel('Average ROI', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

#### Popularity vs. Rating

In [ ]:

plt.figure(figsize=(10, 6))
plt.scatter(movie_df['popularity'], movie_df['rating'], alpha=0.7, color='purple')
plt.title('Popularity vs Rating', fontsize=16)
plt.xlabel('Popularity', fontsize=14)
plt.ylabel('Rating', fontsize=14)
plt.grid(True)
plt.show()

In [ ]:
# Simplify the data by grouping popularity into bins
movie_df['popularity_bin'] = pd.cut(movie_df['popularity'], bins=10, labels=False)

# Calculate the average rating for each popularity bin
popularity_vs_rating = movie_df.groupby('popularity_bin')['rating'].mean().reset_index()

# Plot a bar chart
plt.figure(figsize=(10, 6))
plt.bar(popularity_vs_rating['popularity_bin'], popularity_vs_rating['rating'], color='purple', alpha=0.7)
plt.title('Average Rating by Popularity Bin', fontsize=16)
plt.xlabel('Popularity Bin', fontsize=14)
plt.ylabel('Average Rating', fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

#### Yearly Trends in Box Office Performance

In [ ]:
# Extract year from release_date
movie_df['release_year'] = movie_df['release_date'].dt.year

# Group by year and calculate total revenue and average revenue
yearly_data = movie_df.groupby('release_year').agg(
    total_revenue=('revenue_musd', 'sum'),
    avg_revenue=('revenue_musd', 'mean')
).reset_index()

# Plot yearly trends
plt.figure(figsize=(12, 6))
plt.plot(yearly_data['release_year'], yearly_data['total_revenue'], label='Total Revenue (in M USD)', color='blue')
plt.plot(yearly_data['release_year'], yearly_data['avg_revenue'], label='Average Revenue (in M USD)', color='green')
plt.title('Yearly Trends in Box Office Performance', fontsize=16)
plt.xlabel('Year', fontsize=14)
plt.ylabel('Revenue (in Million USD)', fontsize=14)
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Group by year and calculate total revenue
yearly_revenue = movie_df.groupby('release_year')['revenue_musd'].sum().reset_index()

plt.figure(figsize=(10, 5))
plt.plot(yearly_revenue['release_year'], yearly_revenue['revenue_musd'], marker='o', color='blue')
plt.title('Total Yearly Revenue (in Million USD)', fontsize=14)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Total Revenue (M USD)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# Group by year and calculate total budget
yearly_budget = movie_df.groupby('release_year')['budget_musd'].sum().reset_index()

plt.figure(figsize=(10, 5))
plt.plot(yearly_budget['release_year'], yearly_budget['budget_musd'], marker='o', color='green')
plt.title('Total Yearly Budget (in Million USD)', fontsize=14)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Total Budget (M USD)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# Bar chart for Revenue vs Budget (Top Revenue Movies)
top_revenue_movies = highest_revenue.head(10)  # Select top 10 movies by revenue
plt.figure(figsize=(12, 6))
bar_width = 0.4
x = np.arange(len(top_revenue_movies))

plt.bar(x - bar_width/2, top_revenue_movies['budget_musd'], width=bar_width, label='Budget (M USD)', color='blue')
plt.bar(x + bar_width/2, top_revenue_movies['revenue_musd'], width=bar_width, label='Revenue (M USD)', color='green')

plt.title('Revenue vs Budget (Top Revenue Movies)', fontsize=16)
plt.xlabel('Movies', fontsize=14)
plt.ylabel('Amount (in Million USD)', fontsize=14)
plt.xticks(x, top_revenue_movies['title'], rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Horizontal bar chart for ROI Distribution by Genre
roi_by_genre = highest_roi.explode('genres').groupby('genres')['result'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 8))
roi_by_genre.plot(kind='barh', color='teal', alpha=0.7)
plt.title('ROI Distribution by Genre', fontsize=16)
plt.xlabel('Average ROI', fontsize=14)
plt.ylabel('Genre', fontsize=14)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Stacked bar chart for Yearly Trends in Box Office Performance
yearly_data = movie_df.groupby('release_year').agg(
    total_revenue=('revenue_musd', 'sum'),
    avg_revenue=('revenue_musd', 'mean')
).reset_index()

plt.figure(figsize=(12, 6))
plt.bar(yearly_data['release_year'], yearly_data['total_revenue'], label='Total Revenue (M USD)', color='blue', alpha=0.7)
plt.bar(yearly_data['release_year'], yearly_data['avg_revenue'], label='Average Revenue (M USD)', color='green', alpha=0.7, bottom=yearly_data['total_revenue'])

plt.title('Yearly Trends in Box Office Performance', fontsize=16)
plt.xlabel('Year', fontsize=14)
plt.ylabel('Revenue (in Million USD)', fontsize=14)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()